# Feature Selection 
Shows feature elimination decisions

`results/feature_selection.csv`:
- Logged results in a table

`src/features/transformers.py`:
- Custom transformers implementing engineered features (e.g., lag
features, ratios, domain-driven encodings).

`src/features/preprocessing.py`:
- Main preprocessing pipeline that drops unused features, integrates
engineered features, and can be reused in modeling.

### Description of Merging Procedure

We combined groundwater nitrate measurements with environmental and agricultural datasets to create a single dataset for machine learning. The groundwater dataset served as our starting point, containing **5,280 measurements from 1,993 wells** collected between 2022 and 2025.

First, we linked **soil (SSURGO) and terrain (DEM)** features to groundwater observations using their unique well IDs. We then added **PRISM climate variables** matched to each sampling observation, accounting for the timing of nitrate measurements.

Finally, we added **historical agricultural features** calculated within 500 m and 1,000 m buffers around each well. Crop data were matched by well ID and the year preceding each nitrate measurement (e.g., nitrate from 2023 was matched with crop data from 2022).

We used **left joins** to preserve all original groundwater observations and checked for duplicate records and unmatched data. All 5,280 observations were matched to an agricultural well-year record, although some individual features have missing values due to limited spatial coverage.

The final integrated dataset contains **5,280 observations and 147 columns**, including nitrate concentrations, environmental predictors, agricultural features, and supporting metadata.

## 0. Setup

**Objective:** Load the integrated lag-1 dataset and confirm its dimensions before screening features. Each row is one nitrate sample; repeated wells are handled later through the team's 10 km spatial folds rather than by averaging.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Find the repository root so paths work on any teammate's computer
cwd = Path.cwd()
ROOT = next(p for p in [cwd, *cwd.parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))

DATA_PATH = ROOT / "notebooks" / "merge" / "groundwater_ml_integration_lag1_2021_2024_with_climate.csv"
RESULTS_PATH = ROOT / "results" / "feature_selection.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["gm_samp_collection_date"])

print(f"{len(df):,} rows, {df.shape[1]} columns, {df['gm_well_id'].nunique():,} wells")
print("Samples per year:", df["year"].value_counts().sort_index().to_dict())

5,280 rows, 147 columns, 1,993 wells
Samples per year: {2022: 1455, 2023: 1298, 2024: 1252, 2025: 1275}


## 1. Feature Inventory

**Objective:** Classify all 147 columns into the target, candidate predictor groups (soils, terrain, climate, land use), quality-control/coverage columns, and metadata. This inventory is the starting point for every selection decision that follows.

In [2]:
TARGET = "nitrate_mg_L"

METADATA = [
    "gm_well_id", "gm_well_category", "gm_samp_collection_date", "gm_latitude", "gm_longitude",
    "year", "source_csv_row", "observation_id", "crop_year", "measurement_year",
    "dwr_map_status", "change_base_year", "crop_source_match",
]
TERRAIN = ["elevation_m", "topographic_position"]
CLIMATE_KEYWORDS = ("precip", "temperature")


def feature_group(col):
    """Assign each column to one group based on its name."""
    if col == TARGET:
        return "target"
    if col in METADATA:
        return "metadata"
    if (col.endswith("coverage_pct") or "status" in col or "buffer_area" in col
            or "dwr_polygon_area" in col or "dwr_polygon_footprint_pct" in col):
        return "qc_coverage"
    if col.startswith("soil_"):
        return "soils"
    if col in TERRAIN:
        return "terrain"
    if any(k in col for k in CLIMATE_KEYWORDS):
        return "climate"
    if col.startswith("delta_"):
        return "land_use_delta"
    if col.endswith("__previous"):
        return "land_use_previous"
    return "land_use_current"


inventory = pd.DataFrame({"feature": df.columns})
inventory["group"] = inventory["feature"].map(feature_group)

inventory["group"].value_counts()

group
land_use_previous    34
land_use_current     34
qc_coverage          24
land_use_delta       18
soils                15
metadata             13
climate               6
terrain               2
target                1
Name: count, dtype: int64

## 2. Domain and Metadata Exclusion

**Objective:** Remove columns that are not environmental predictors (identifiers, dates, bookkeeping, raw coordinates, and data-quality/coverage columns) and the t-2 land-use columns, which are fully redundant with current land use plus annual change. Well category (domestic vs. irrigation/industrial) is retained as a candidate. Every decision is logged as `feature, step, decision, reason`.

In [3]:
decisions = []


def log_decision(features, step, decision, reason):
    """Record one selection decision per feature."""
    for f in features:
        decisions.append({"feature": f, "step": step, "decision": decision, "reason": reason})


by_group = inventory.groupby("group")["feature"].apply(list)

COORDINATES = ["gm_latitude", "gm_longitude"]
WELL_ATTRIBUTES = ["gm_well_category"]  # kept as a candidate predictor
other_metadata = [f for f in METADATA if f not in COORDINATES + WELL_ATTRIBUTES]

log_decision(other_metadata, "domain", "drop",
             "identifier, date or bookkeeping column; not an environmental predictor")
log_decision(COORDINATES, "domain", "drop",
             "raw coordinates; location alone showed no skill under 10 km spatial folds, "
             "and coordinates can let the model memorize areas instead of learning from land conditions")
log_decision(by_group["qc_coverage"], "domain", "drop",
             "data-quality or coverage column; documents processing, not land conditions")
log_decision(by_group["land_use_previous"], "redundancy", "drop",
             "t-2 land use; equals current minus delta exactly, so current + delta keep the same information")

log = pd.DataFrame(decisions)
candidates = [f for f in df.columns if f not in set(log["feature"]) and f != TARGET]

print(log.groupby(["step", "decision"]).size())
print(f"\nCandidates remaining: {len(candidates)}")

step        decision
domain      drop        36
redundancy  drop        34
dtype: int64

Candidates remaining: 76


## 3. Soil Class Consolidation

**Objective:** Combine sparse soil classes into meaningful groups before screening for low information. Seven drainage classes are grouped by drainage speed (rapid, moderate, slow), and the dual hydrologic group C/D is merged into D, following the USDA convention that C/D soils behave as D when undrained. 

Grouping preserves the signal of rare classes (e.g., very poorly drained soils, present at only 7 wells) instead of discarding them.

In [4]:
DRAINAGE_GROUPS = {
    "soil_drainage_rapid_pct": ["excessively_drained", "somewhat_excessively_drained"],
    "soil_drainage_moderate_pct": ["well_drained", "moderately_well_drained"],
    "soil_drainage_slow_pct": ["somewhat_poorly_drained", "poorly_drained", "very_poorly_drained"],
}
HYDGROUP_D = ["soil_hydgrpdcd_pct_c_d", "soil_hydgrpdcd_pct_d"]

new_cols = {}

for new_col, classes in DRAINAGE_GROUPS.items():
    original = [f"soil_drclassdcd_pct_{c}" for c in classes]
    new_cols[new_col] = df[original].sum(axis=1, min_count=1)
    log_decision(original, "domain", "combine", f"merged into {new_col} (grouped by drainage speed)")
    candidates = [f for f in candidates if f not in original] + [new_col]

new_cols["soil_hydgrp_d_pct"] = df[HYDGROUP_D].sum(axis=1, min_count=1)
log_decision(HYDGROUP_D, "domain", "combine",
             "merged into soil_hydgrp_d_pct; C/D soils behave as D when undrained (USDA convention)")
candidates = [f for f in candidates if f not in HYDGROUP_D] + ["soil_hydgrp_d_pct"]

df = pd.concat([df, pd.DataFrame(new_cols)], axis=1)

# Check: the grouped classes should still add up to 100% of the mapped soil area
drainage_total = df[list(DRAINAGE_GROUPS)].sum(axis=1)
hydgroup_total = df[["soil_hydgrpdcd_pct_a", "soil_hydgrpdcd_pct_b",
                     "soil_hydgrpdcd_pct_c", "soil_hydgrp_d_pct"]].sum(axis=1, min_count=1)

print(f"Drainage groups sum to: {drainage_total.min():.1f} - {drainage_total.max():.1f}")
print(f"Hydrologic groups sum to: {hydgroup_total.min():.1f} - {hydgroup_total.max():.1f}")
print(f"Candidates remaining: {len(candidates)}")

Drainage groups sum to: 100.0 - 100.0
Hydrologic groups sum to: 100.0 - 100.0
Candidates remaining: 71


## 4. Low Information

**Objective:** Remove candidates with more than 50% missing values or with a single value in more than 95% of rows. This screen does not use nitrate, so it can be applied to the full dataset without leakage.

In [5]:
MISSING_THRESHOLD = 0.50
DOMINANT_THRESHOLD = 0.95

missing_share = df[candidates].isna().mean()
dominant_share = df[candidates].apply(lambda s: s.value_counts(normalize=True).iloc[0])

too_missing = missing_share[missing_share > MISSING_THRESHOLD].index.tolist()
near_constant = dominant_share[dominant_share > DOMINANT_THRESHOLD].index.tolist()

log_decision(too_missing, "low_information", "drop",
             f"more than {MISSING_THRESHOLD:.0%} missing values")
log_decision(near_constant, "low_information", "drop",
             f"one value in more than {DOMINANT_THRESHOLD:.0%} of rows")

candidates = [f for f in candidates if f not in too_missing + near_constant]

print("Too many missing:", {f: round(missing_share[f], 2) for f in too_missing})
print("Near-constant:", {f: round(dominant_share[f], 3) for f in near_constant})
print(f"Candidates remaining: {len(candidates)}")

Too many missing: {'soil_brockdepmin': np.float64(0.73)}
Near-constant: {'delta_urban_pp_500m': np.float64(0.964)}
Candidates remaining: 69


## 5. Redundancy

**Objective:** Remove duplicated information without using nitrate. (1) Land-use shares are expressed as a percentage of the whole buffer (`pct`), so the `_of_mapped` versions are dropped; the unmapped share is kept through `outside_dwr_polygons_pct`. (2) When the same variable at 500 m and 1,000 m has Spearman ρ > 0.9, only the 500 m version is kept, matching the soil buffer scale and the area closest to the well. Correlations across different variables (e.g., crop intensity vs. elevation) are not used to drop features; their added value is assessed in the model-based step.

In [6]:
RADIUS_THRESHOLD = 0.90

# 1. Keep pct (share of whole buffer); drop the share-of-mapped-area versions
of_mapped = [f for f in candidates if "_of_mapped_" in f]
log_decision(of_mapped, "redundancy", "drop",
             "share of DWR-mapped area; pct (share of whole buffer) is used instead, "
             "with outside_dwr_polygons_pct describing the unmapped share")
candidates = [f for f in candidates if f not in of_mapped]

# 2. Compare each variable at 500 m vs 1,000 m
pairs = [(f, f.replace("500m", "1000m")) for f in candidates
         if "500m" in f and f.replace("500m", "1000m") in candidates]

radius = pd.DataFrame(
    [{"feature_500m": a, "feature_1000m": b,
      "spearman": df[[a, b]].corr(method="spearman").iloc[0, 1]} for a, b in pairs]
).sort_values("spearman", ascending=False)

redundant_1000m = radius.loc[radius["spearman"] > RADIUS_THRESHOLD, "feature_1000m"].tolist()
log_decision(redundant_1000m, "redundancy", "drop",
             f"Spearman rho > {RADIUS_THRESHOLD} with the 500 m version; 500 m kept")
candidates = [f for f in candidates if f not in redundant_1000m]

display(radius.round(2))
print(f"Candidates remaining: {len(candidates)}")

,feature_500m,feature_1000m,spearman
8,pct_multicrop_500m,pct_multicrop_1000m,0.95
0,pct_veg_berry_500m,pct_veg_berry_1000m,0.95
7,crop_cycles_500m,crop_cycles_1000m,0.94
1,pct_vineyard_500m,pct_vineyard_1000m,0.94
6,pct_cropland_500m,pct_cropland_1000m,0.90
9,outside_dwr_polygons_pct_500m,outside_dwr_polygons_pct_1000m,0.89
2,pct_orchard_500m,pct_orchard_1000m,0.77
4,pct_idle_fallow_500m,pct_idle_fallow_1000m,0.77
3,pct_grain_field_pasture_500m,pct_grain_field_pasture_1000m,0.71
5,pct_urban_500m,pct_urban_1000m,0.71


Candidates remaining: 50


## 6. Selected Features

**Objective:** Combine every decision into one table covering all original columns plus the engineered soil groups: group, step, decision, reason, and Spearman correlation with nitrate (descriptive only, full dataset). The table is exported to `results/feature_selection.csv`.

In [7]:
selected = candidates.copy()

new_features = pd.DataFrame({"feature": list(DRAINAGE_GROUPS) + ["soil_hydgrp_d_pct"], "group": "soils"})
all_features = pd.concat([inventory, new_features], ignore_index=True)

log = pd.DataFrame(decisions)
feature_table = all_features.merge(log, on="feature", how="left")
feature_table.loc[feature_table["feature"].isin(selected), ["step", "decision", "reason"]] = [
    "final", "keep", "passed domain, low-information and redundancy screening"]
feature_table.loc[feature_table["feature"] == TARGET, ["step", "decision"]] = ["target", "target"]

feature_table["spearman_with_nitrate"] = feature_table["feature"].map(
    lambda f: df[f].corr(df[TARGET], method="spearman")
    if f != TARGET and pd.api.types.is_numeric_dtype(df[f]) else np.nan)

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
feature_table.to_csv(RESULTS_PATH, index=False)

display(pd.crosstab(feature_table["group"], feature_table["decision"]))
print(f"Saved {len(feature_table)} rows to {RESULTS_PATH.relative_to(ROOT)}")
print(f"Selected features: {len(selected)}")

decision,combine,drop,keep,target
group,,,,
climate,0,0,6,0
land_use_current,0,19,15,0
land_use_delta,0,1,17,0
land_use_previous,0,34,0,0
metadata,0,12,1,0
qc_coverage,0,24,0,0
soils,9,1,9,0
target,0,0,0,1
terrain,0,0,2,0


Saved 151 rows to results\feature_selection.csv
Selected features: 50


## 7. Conclusions

**Summary.** The integrated dataset (5,280 nitrate samples, 1,993 wells, 147 columns) was screened to **50 candidate features for modeling** through four model-independent steps: domain and metadata exclusion (147 → 76), soil class consolidation (→ 71), low-information screening (→ 69), and redundancy removal (→ 50). No step used nitrate values, so the screening carries no target leakage. All decisions are logged in `results/feature_selection.csv`.

**Key decisions and rationale**
- **Unit of analysis:** each nitrate sample is one row. Samples are not averaged per well because crop (lag-1) and antecedent climate features change from year to year; averaging would remove the temporal information these features carry. Repeated and nearby wells will be handled through the team's 10 km spatial folds.
- **Coordinates excluded:** a location-only model showed no skill under 10 km spatial folds, and raw coordinates could let the model memorize areas instead of learning from land conditions.
- **Well category retained:** domestic vs. irrigation/industrial wells may differ in depth, but well depth is not available to confirm this.
- **t−2 land use removed:** current land use minus the t−2 value equals the annual change exactly, so current land use plus annual change keep all the information.
- **Soil classes consolidated:** seven drainage classes were grouped by drainage speed (rapid, moderate, slow), and hydrologic group C/D was merged into D (USDA convention), preserving the signal of rare classes such as very poorly drained soils (7 wells).
- **Low information:** depth to bedrock (73% missing) and 500 m urban change (zero in 96% of rows, possibly a mapping artifact) were removed.
- **Land-use definition:** shares are expressed as a percentage of the whole buffer (`pct`), with `outside_dwr_polygons_pct` describing the unmapped share. When the 500 m and 1,000 m versions of a variable had Spearman ρ > 0.9, the 500 m version was kept.
- **Correlations across variable types were not used to drop features.** Agricultural intensity, elevation, slope, and precipitation overlap strongly (see EDA), and which of them adds predictive value should be decided by model evaluation, not by pairwise correlation.

**Next steps for modeling**
- Train tree-based models (e.g., decision tree, random forest) on the 50 candidates using the 10 km spatial folds and the space-time split.
- Assess feature contributions with grouped permutation importance (soils, terrain, climate, agricultural intensity, land-use change) to account for correlated predictors.
- Compare the full 50-feature set with reduced sets by cross-validated RMSE.

**Caveats**
- Drainage groups and hydrologic groups each sum to 100%, so one group in each set is redundant by construction; this matters for linear models but not for tree-based models.
- Spearman correlations in the results table use the full dataset and are descriptive only; they were not used for selection.
- The 2024 DWR crop map is provisional and feeds the 2025 test year of the space-time evaluation. The one-year crop lag is a modeling assumption, not an established nitrate transport time.

## 8. Feature Dictionary — Modeling Features

| Group | Feature | Description |
|---|---|---|
| Target | `nitrate_mg_L` | Groundwater nitrate concentration (mg/L); modeled as log(1 + nitrate) |
| Soils | `soil_slopegradwta` | Area-weighted soil slope gradient (%) within the well buffer (SSURGO) |
| Soils | `soil_aws0150wta` | Area-weighted available water storage, 0–150 cm (SSURGO) |
| Soils | `soil_drainage_rapid_pct` | % of buffer soils excessively or somewhat excessively drained |
| Soils | `soil_drainage_moderate_pct` | % of buffer soils well or moderately well drained |
| Soils | `soil_drainage_slow_pct` | % of buffer soils somewhat poorly, poorly or very poorly drained |
| Soils | `soil_hydgrpdcd_pct_a` | % of buffer in hydrologic group A (high infiltration) |
| Soils | `soil_hydgrpdcd_pct_b` | % of buffer in hydrologic group B (moderate infiltration) |
| Soils | `soil_hydgrpdcd_pct_c` | % of buffer in hydrologic group C (slow infiltration) |
| Soils | `soil_hydgrp_d_pct` | % of buffer in hydrologic group D, including C/D (very slow infiltration) |
| Terrain | `elevation_m` | Ground elevation at the well (10 m DEM) |
| Terrain | `topographic_position` | Well elevation minus mean elevation within 500 m |
| Climate | `historical_mean_annual_precip_mm` | Mean annual precipitation, complete years before the sample year (PRISM) |
| Climate | `previous_3_complete_months_precip_mm` | Total precipitation, 3 complete months before the sample month (PRISM) |
| Climate | `previous_12_complete_months_precip_mm` | Total precipitation, 12 complete months before the sample month (PRISM) |
| Climate | `historical_mean_temperature_c` | Mean temperature, complete years before the sample year (PRISM) |
| Climate | `previous_3_complete_months_temperature_c` | Mean temperature, 3 complete months before the sample month (PRISM) |
| Climate | `previous_12_complete_months_temperature_c` | Mean temperature, 12 complete months before the sample month (PRISM) |
| Well | `gm_well_category` | Well type: domestic vs. irrigation/industrial |
| Land use | `pct_veg_berry_500m` | % of 500 m buffer in vegetables and berries (crop year t−1) |
| Land use | `pct_vineyard_500m` | % of 500 m buffer in vineyards (crop year t−1) |
| Land use | `pct_orchard_500m` | % of 500 m buffer in orchards (crop year t−1) |
| Land use | `pct_orchard_1000m` | % of 1,000 m buffer in orchards (crop year t−1) |
| Land use | `pct_grain_field_pasture_500m` | % of 500 m buffer in grain, field crops, hay and pasture (crop year t−1) |
| Land use | `pct_grain_field_pasture_1000m` | % of 1,000 m buffer in grain, field crops, hay and pasture (crop year t−1) |
| Land use | `pct_idle_fallow_500m` | % of 500 m buffer in idle or fallow land (crop year t−1) |
| Land use | `pct_idle_fallow_1000m` | % of 1,000 m buffer in idle or fallow land (crop year t−1) |
| Land use | `pct_urban_500m` | % of 500 m buffer in urban land (crop year t−1) |
| Land use | `pct_urban_1000m` | % of 1,000 m buffer in urban land (crop year t−1) |
| Land use | `pct_cropland_500m` | % of 500 m buffer in all cropland (crop year t−1) |
| Land use | `pct_multicrop_500m` | % of 500 m buffer cropped two or more times (crop year t−1) |
| Land use | `crop_cycles_500m` | Mean crop cycles per year on farmland within 500 m (crop year t−1) |
| Land use | `outside_dwr_polygons_pct_500m` | % of 500 m buffer outside DWR-mapped polygons (unmapped area) |
| Land use | `outside_dwr_polygons_pct_1000m` | % of 1,000 m buffer outside DWR-mapped polygons (unmapped area) |
| Land-use change | `delta_veg_berry_pp_500m` | Change in % vegetables and berries within 500 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_veg_berry_pp_1000m` | Change in % vegetables and berries within 1,000 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_vineyard_pp_500m` | Change in % vineyards within 500 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_vineyard_pp_1000m` | Change in % vineyards within 1,000 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_orchard_pp_500m` | Change in % orchards within 500 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_orchard_pp_1000m` | Change in % orchards within 1,000 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_grain_field_pasture_pp_500m` | Change in % grain, field crops, hay and pasture within 500 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_grain_field_pasture_pp_1000m` | Change in % grain, field crops, hay and pasture within 1,000 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_idle_fallow_pp_500m` | Change in % idle or fallow land within 500 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_idle_fallow_pp_1000m` | Change in % idle or fallow land within 1,000 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_urban_pp_1000m` | Change in % urban land within 1,000 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_cropland_pp_500m` | Change in % all cropland within 500 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_cropland_pp_1000m` | Change in % all cropland within 1,000 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_multicrop_pp_500m` | Change in % multicrop land within 500 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_multicrop_pp_1000m` | Change in % multicrop land within 1,000 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_dwr_polygon_footprint_pp_500m` | Change in DWR-mapped share of 500 m buffer, t−1 vs. t−2 (pp) |
| Land-use change | `delta_dwr_polygon_footprint_pp_1000m` | Change in DWR-mapped share of 1,000 m buffer, t−1 vs. t−2 (pp) |

The full record of all 151 columns (147 original + 4 engineered soil groups), with the step, decision, and reason for each, is in `results/feature_selection.csv`.

## 9. Feature Dictionary — Selected Features

| Group | Feature | Description |
|---|---|---|
| Target | `nitrate_mg_L` | Groundwater nitrate concentration (mg/L); modeled as log(1 + nitrate) |
| Soils | `soil_slopegradwta` | Area-weighted soil slope gradient (%) within the well buffer (SSURGO) |
| Soils | `soil_aws0150wta` | Area-weighted available water storage, 0–150 cm (SSURGO) |
| Soils | `soil_drainage_moderate_pct` | % of buffer soils well or moderately well drained |
| Soils | `soil_drainage_slow_pct` | % of buffer soils somewhat poorly, poorly or very poorly drained |
| Soils | `soil_hydgrpdcd_pct_a` | % of buffer in hydrologic group A (high infiltration) |
| Soils | `soil_hydgrpdcd_pct_b` | % of buffer in hydrologic group B (moderate infiltration) |
| Soils | `soil_hydgrpdcd_pct_c` | % of buffer in hydrologic group C (slow infiltration) |
| Soils | `soil_hydgrp_d_pct` | % of buffer in hydrologic group D, including C/D (very slow infiltration) |
| Terrain | `topographic_position` | Well elevation minus mean elevation within 500 m |
| Climate | `historical_mean_annual_precip_mm` | Mean annual precipitation, complete years before the sample year (PRISM) |
| Climate | `previous_3_complete_months_precip_mm` | Total precipitation, 3 complete months before the sample month (PRISM) |
| Climate | `previous_12_complete_months_precip_mm` | Total precipitation, 12 complete months before the sample month (PRISM) |
| Climate | `historical_mean_temperature_c` | Mean temperature, complete years before the sample year (PRISM) |
| Climate | `previous_3_complete_months_temperature_c` | Mean temperature, 3 complete months before the sample month (PRISM) |
| Climate | `previous_12_complete_months_temperature_c` | Mean temperature, 12 complete months before the sample month (PRISM) |
| Well | `gm_well_category` | Well type: domestic vs. irrigation/industrial (1 = domestic) |
| Land use | `pct_veg_berry_500m` | % of 500 m buffer in vegetables and berries (crop year t−1) |
| Land use | `pct_vineyard_500m` | % of 500 m buffer in vineyards (crop year t−1) |
| Land use | `pct_orchard_500m` | % of 500 m buffer in orchards (crop year t−1) |
| Land use | `pct_orchard_1000m` | % of 1,000 m buffer in orchards (crop year t−1) |
| Land use | `pct_grain_field_pasture_500m` | % of 500 m buffer in grain, field crops, hay and pasture (crop year t−1) |
| Land use | `pct_grain_field_pasture_1000m` | % of 1,000 m buffer in grain, field crops, hay and pasture (crop year t−1) |
| Land use | `pct_idle_fallow_500m` | % of 500 m buffer in idle or fallow land (crop year t−1) |
| Land use | `pct_urban_500m` | % of 500 m buffer in urban land (crop year t−1) |
| Land use | `pct_multicrop_500m` | % of 500 m buffer cropped two or more times (crop year t−1) |
| Land-use change | `delta_veg_berry_pp_1000m` | Change in % vegetables and berries within 1,000 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_vineyard_pp_500m` | Change in % vineyards within 500 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_orchard_pp_1000m` | Change in % orchards within 1,000 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_urban_pp_1000m` | Change in % urban land within 1,000 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_cropland_pp_1000m` | Change in % all cropland within 1,000 m, t−1 vs. t−2 (pp) |
| Land-use change | `delta_multicrop_pp_500m` | Change in % multicrop land within 500 m, t−1 vs. t−2 (pp) |

The full record of all 151 columns (147 original + 4 engineered soil groups), with the step, decision, and reason for each, is in `results/feature_selection.csv`.